# Praktikum 10 – RAG-Evaluation und Benchmarking
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Ein größeres Golden Dataset aufbauen, LLM-as-a-Judge mit einer zweiten Metrik vergleichen und Antworten nach Fehlerklassen einordnen.

**Zielprodukt nach 90 Minuten:**
1. Ein kleines Benchmark-Dataset mit mehreren Falltypen.
2. Eine Auswertung mit Judge-Score, Jaccard und Faithfulness.
3. Eine kurze Fehleranalyse statt nur eines Mittelwerts.

In [ ]:
import importlib
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
    "pandas": ("pandas", "3.0.2"),
}



def run_install(specs):
    commands = []

    if shutil.which("uv") is not None:
        commands.append(["uv", "pip", "install", "--python", sys.executable, *specs])

    commands.append([sys.executable, "-m", "pip", "install", *specs])
    if not IN_COLAB:
        commands.append([sys.executable, "-m", "pip", "install", "--user", *specs])

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    install_name
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import pandas as pd
import requests

DEFAULT_OPTIONS = {
    "num_ctx": 8192,
    "num_predict": 512,
    "temperature": 0.5,
}


def build_options(**overrides):
    options = DEFAULT_OPTIONS.copy()
    for key, value in overrides.items():
        if value is not None:
            options[key] = value
    return options


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if sys.platform.startswith("win"):
        raise RuntimeError("Ollama fehlt unter Windows. Installiere es über https://ollama.com/download/windows und führe die Setup-Zelle erneut aus.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    log_path = os.path.join(tempfile.gettempdir(), "ollama-notebook.log")
    ollama_log = open(log_path, "a", encoding="utf-8")
    popen_kwargs = {"stdout": ollama_log, "stderr": subprocess.STDOUT}
    if os.name != "nt":
        popen_kwargs["start_new_session"] = True
    subprocess.Popen(["ollama", "serve"], **popen_kwargs)
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
model_aliases = {MODEL, f"{MODEL}:latest"}
if not any(name in model_aliases for name in available_models):
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))


## Teil 1 – Golden Dataset mit mehreren Falltypen (25 min)
Wir erzeugen ein kleines, aber abwechslungsreiches Benchmark-Set mit beantwortbaren und nicht beantwortbaren Fragen.

**Pflichtschritte:**
- Nutze mindestens 10 Beispiele.
- Markiere, ob die Frage aus dem Kontext beantwortbar ist.
- Erzeuge für alle Beispiele Modellantworten mit demselben Prompt.

**Soll-Ergebnis:** eine Tabelle mit Fragen, Kontexten, Ground Truth und Modellantworten.

In [ ]:
test_set = [
    {"category": "fakt", "answerable": True, "q": "Wie hoch ist der Mount Everest?", "c": "Der Mount Everest ist 8848 Meter hoch.", "gt": "8848 Meter"},
    {"category": "fakt", "answerable": True, "q": "Wer schrieb Faust?", "c": "Goethe verfasste das Drama Faust.", "gt": "Johann Wolfgang von Goethe"},
    {"category": "ml", "answerable": True, "q": "Was machen Embeddings?", "c": "Embeddings bilden Texte als Vektoren ab, damit Ähnlichkeiten berechnet werden können.", "gt": "Sie bilden Texte als Vektoren ab"},
    {"category": "ml", "answerable": True, "q": "Was ist Faithfulness?", "c": "Faithfulness bedeutet, dass eine Antwort vollständig durch den gegebenen Kontext gedeckt ist.", "gt": "Eine Antwort ist vollständig durch den Kontext gedeckt"},
    {"category": "systeme", "answerable": True, "q": "Welche Plattform führt Java typischerweise aus?", "c": "Java-Programme laufen typischerweise auf der Java Virtual Machine, kurz JVM.", "gt": "Java Virtual Machine"},
    {"category": "security", "answerable": True, "q": "Wozu dient ein Moderations-Layer?", "c": "Ein Moderations-Layer filtert problematische Eingaben oder Ausgaben, bevor ein Modell antwortet.", "gt": "Er filtert problematische Eingaben oder Ausgaben"},
    {"category": "fakt", "answerable": False, "q": "Wann wurde der Mount Everest erstmals bestiegen?", "c": "Der Mount Everest ist 8848 Meter hoch.", "gt": "Unbekannt"},
    {"category": "ml", "answerable": False, "q": "Welche Vektordatenbank wurde im Kontext konkret verwendet?", "c": "Embeddings bilden Texte als Vektoren ab, damit Ähnlichkeiten berechnet werden können.", "gt": "Unbekannt"},
    {"category": "security", "answerable": False, "q": "Wie heisst das verwendete Moderationsmodell?", "c": "Ein Moderations-Layer filtert problematische Eingaben oder Ausgaben, bevor ein Modell antwortet.", "gt": "Unbekannt"},
    {"category": "systeme", "answerable": True, "q": "Warum sind relationale Datenbanken stark?", "c": "Relationale Datenbanken eignen sich gut für strukturierte Tabellen, Joins und Transaktionen.", "gt": "Für strukturierte Tabellen, Joins und Transaktionen"},
]


def get_answer(query, context):
    prompt = (
        "Beantworte die Frage ausschliesslich mit Informationen aus dem Kontext. "
        "Wenn der Kontext die Information nicht enthält, antworte exakt mit Unbekannt. "
        "Antworte kurz und präzise in einem Satz.\n\n"
        f"Kontext: {context}\nFrage: {query}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options=build_options(temperature=0, num_predict=40, stop=["\n\n"]),
        keep_alive="10m",
    )
    return response["message"]["content"].strip()


results = []
for item in test_set:
    answer = get_answer(item["q"], item["c"])
    results.append(
        {
            "category": item["category"],
            "answerable": item["answerable"],
            "query": item["q"],
            "context": item["c"],
            "answer": answer,
            "ground_truth": item["gt"],
        }
    )

df = pd.DataFrame(results)
print(df[["category", "answerable", "query", "answer", "ground_truth"]].to_string(index=False))

## Teil 2 – Judge-Score und zweite Referenzmetrik (35 min)
Wir vergleichen LLM-as-a-Judge mit einer einfachen tokenbasierten Überschneidungsmetrik.

**Pflichtschritte:**
- Vergib für jede Antwort einen Judge-Score zwischen 0 und 1.
- Berechne zusätzlich einen Jaccard-Score.
- Vergleiche danach, wo beide Metriken übereinstimmen oder auseinanderlaufen.

**Soll-Ergebnis:** zwei Kennzahlen pro Antwort statt einer Einzelmeinung des Judges.

In [ ]:
def extract_judge_score(text):
    matches = re.findall(r"\d+(?:\.\d+)?", text)
    if not matches:
        raise RuntimeError(f"Judge output does not contain a numeric score: {text!r}")

    score = float(matches[0])
    if not 0.0 <= score <= 1.0:
        raise RuntimeError(f"Judge score must be between 0.0 and 1.0, got {score}")
    return score


def judge_answer(query, answer, ground_truth):
    prompt = f"""Bewerte die Korrektheit der Antwort basierend auf der Ground Truth.
Frage: {query}
Antwort: {answer}
Ground Truth: {ground_truth}

Gib einen Score von 0 bis 1. Antworte NUR mit der Zahl.
"""
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options=build_options(temperature=0, num_predict=8, stop=["\n"]),
        keep_alive="10m",
    )
    return extract_judge_score(response["message"]["content"])


def normalize_tokens(text):
    return {token for token in re.findall(r"\w+", text.lower()) if token}


def jaccard_score(answer, ground_truth):
    answer_tokens = normalize_tokens(answer)
    truth_tokens = normalize_tokens(ground_truth)
    if not answer_tokens and not truth_tokens:
        return 1.0
    if not answer_tokens or not truth_tokens:
        return 0.0
    return len(answer_tokens & truth_tokens) / len(answer_tokens | truth_tokens)


df["judge_score"] = df.apply(
    lambda row: judge_answer(row["query"], row["answer"], row["ground_truth"]), axis=1
)
df["jaccard"] = df.apply(lambda row: jaccard_score(row["answer"], row["ground_truth"]), axis=1)

print(df[["query", "answer", "ground_truth", "judge_score", "jaccard"]].to_string(index=False))

## Teil 3 – Faithfulness und Fehlerklassen (30 min)
Wir unterscheiden zwischen falschen Antworten, fehlendem Kontext und nicht kontexttreuen Antworten.

**Pflichtschritte:**
- Führe für alle Fälle einen Faithfulness-Check durch.
- Weise danach jeder Antwort eine Fehlerklasse zu.
- Vergleiche die Verteilung der Fehlerklassen zwischen beantwortbaren und unbeantwortbaren Fragen.

**Soll-Ergebnis:** eine kleine Fehleranalyse statt nur eines Durchschnittsscores.

In [ ]:
def check_faithfulness(context, answer):
    prompt = (
        "Enthält die Antwort Informationen, die nicht im Kontext stehen? "
        "Antworte exakt mit JA oder NEIN.\n\n"
        f"Kontext: {context}\nAntwort: {answer}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options=build_options(temperature=0, num_predict=8, stop=["\n"]),
        keep_alive="10m",
    )
    return response["message"]["content"].strip()


def classify_case(row):
    if row["ground_truth"] == "Unbekannt":
        if row["answer"].strip() == "Unbekannt" and row["faithfulness"] == "NEIN":
            return "sauber_als_unbekannt_markiert"
        if row["faithfulness"] == "JA":
            return "halluzination_ohne_kontext"
        return "unklarer_unbekannt_fall"

    if row["judge_score"] >= 0.8 and row["faithfulness"] == "NEIN":
        return "sauber_beantwortet"
    if row["judge_score"] < 0.8 and row["faithfulness"] == "NEIN":
        return "inhaltlich_schwach_trotz_kontext"
    if row["judge_score"] >= 0.8 and row["faithfulness"] == "JA":
        return "richtig_aber_nicht_kontexttreu"
    return "fehlerhaft_und_nicht_kontexttreu"


df["faithfulness"] = df.apply(lambda row: check_faithfulness(row["context"], row["answer"]), axis=1)
df["error_type"] = df.apply(classify_case, axis=1)

print(df[["answerable", "query", "judge_score", "jaccard", "faithfulness", "error_type"]].to_string(index=False))

error_counts = df.groupby(["answerable", "error_type"]).size()
print("\nFehlerverteilung:")
print(error_counts.to_string())

## Abgabe und Reflexion
**Pflichtabgabe:**
1. Dokumentiere den Mittelwert von Judge-Score und Jaccard über alle Fälle.
2. Zeige die Verteilung der Fehlerklassen für beantwortbare und unbeantwortbare Fragen.
3. Begründe in 3 bis 5 Sätzen, wo Judge und Jaccard dasselbe signalisieren und wo nicht.

**Transferaufgaben:**
1. Erweitere das Benchmark-Set um mindestens drei adversariale oder mehrdeutige Beispiele.
2. Führe eine dritte Metrik ein, zum Beispiel Exact Match oder Token-Recall.
3. Diskutiere, wann ein LLM-as-a-Judge selbst zur Fehlerquelle wird.